<a href="https://colab.research.google.com/github/pythoncasper-prog/Conversational-Chatbot/blob/main/Conversational_Chatbot_with_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **⚙️ 1️⃣ Installing Libraries**

In [ ]:
!pip install langchain langchain-community langchain-huggingface transformers accelerate torch faiss-cpu sentence-transformers pypdf fitz

# **🧠2️⃣ Importing Required Libraries**



In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline

# Module to create vector database
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

# **🏷 3️⃣ Selecting the Model**

In [ ]:
MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"

# **🔤 4️⃣ Loading the Tokenizer**

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

# **🧠 5️⃣ Loading the Model**

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

# **⚙️ 6️⃣ Creating the Text Generation Pipeline**

In [ ]:
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=300,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.1,
    do_sample=True
)

Passing `generation_config` together with generation-related arguments=({'do_sample', 'repetition_penalty', 'temperature', 'max_new_tokens', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


# **🔗 7️⃣ Wrapping Model in LangChain**

In [ ]:
llm = HuggingFacePipeline(pipeline=pipe)

# **🧠 8️⃣ Creating Vector Database**

### **🔹 Step 1: Load Documents**

In [ ]:
PDF_PATH_NEW = '/content/PDF/Company_Policies_and_Procedures.pdf'
loader_new = PyPDFLoader(PDF_PATH_NEW)

new_documents = loader_new.load()
print(f"Number of documents loaded: {len(new_documents)}")

Number of documents loaded: 2


### **🔹 Step 2: Chunking**

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

chunks = text_splitter.split_documents(new_documents)

6


### **🔹 Step 3: Create Embeddings**

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### **🔹 Step 4: Store in Vector DB**

In [ ]:
vector_store = FAISS.from_documents(documents=chunks,embedding=embeddings)
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

# **🔄 9️⃣ Creating Conversation Chain**

In [ ]:
conversation = ConversationalRetrievalChain.from_llm(
    llm,
    retriever=retriever,
    verbose=True
)

# **🔁 🔟 Chat Loop**


In [ ]:
chat_history = []

print("RAG Chatbot Ready! (type 'exit' to stop)\n")

while True:
    user_input = input("You: ")

    if user_input.lower() == "exit":
        break

    # 1️⃣ Retrieve relevant chunks from RAG
    docs = retriever.invoke(user_input)

    context = "\n\n".join([doc.page_content for doc in docs])


    # 2️⃣ Convert chat history array to text
    history_text = ""

    for chat in chat_history:
        history_text += f"User: {chat['user']}\n"
        history_text += f"Assistant: {chat['assistant']}\n"


    # 3️⃣ Build prompt
    prompt = f"""
You are a helpful assistant.

Answer the question using ONLY the provided context.
If the answer is not in the context, say "I don't know".

Conversation History:
{history_text}

Context:
{context}

Question:
{user_input}

Answer:
"""


    # 4️⃣ Generate response
    response = llm.invoke(prompt)
    answer = response.replace(prompt, "").strip()
    print("Bot:", answer)


    # 5️⃣ Store interaction in array memory
    chat_history.append({
        "user": user_input,
        "assistant": answer
    })

RAG Chatbot Ready! (type 'exit' to stop)

